In [ ]:
import pandas as pd

# ===============================
# 1️⃣ Load CSV files
# ===============================
actual_prices = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\actual_room_price.csv')
predicted_prices = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\model\predicted_room_prices_2025.csv')
booking_data = pd.read_csv(r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\booking_history.csv')

# ===============================
# 2️⃣ Merge booking with prices
# ===============================
def merge_with_booking(price_df):
    return pd.merge(
        booking_data,
        price_df,
        on=['Month', 'Room_Type'],
        how='left'
    )

actual_df = merge_with_booking(actual_prices)
predicted_df = merge_with_booking(predicted_prices)

# ===============================
# 3️⃣ Calculate Revenue & Profit
# ===============================
def calculate_revenue_profit(df):
    if 'Base_Price' not in df.columns:
        BASE_PRICE_MAP = {
            'Standard': 70,
            'Deluxe': 100,
            'Suite': 150
        }
        df['Base_Price'] = df['Room_Type'].map(BASE_PRICE_MAP)

    df['Revenue'] = (
        df['Room_per_night_price'] *
        df['Booked_Rooms'] *
        df['Average_Stay_Days']
    )

    df['Profit'] = (
        (df['Room_per_night_price'] - df['Base_Price']) *
        df['Booked_Rooms'] *
        df['Average_Stay_Days']
    )

    return df.groupby('Month').agg(
        Revenue=('Revenue', 'sum'),
        Profit=('Profit', 'sum')
    ).reset_index()

    return monthly

actual_monthly = calculate_revenue_profit(actual_df)
predicted_monthly = calculate_revenue_profit(predicted_df)

# ===============================
# 4️⃣ Compare Actual vs AI
# ===============================
comparison = pd.merge(
    actual_monthly,
    predicted_monthly,
    on='Month',
    suffixes=('_Actual', '_Predicted')
)

comparison['Profit_Difference'] = (
    comparison['Profit_Predicted'] - comparison['Profit_Actual']
)

comparison['Revenue_Difference'] = (
    comparison['Revenue_Predicted'] - comparison['Revenue_Actual']
)

# ===============================
# 5️⃣ Sort months correctly
# ===============================
month_order = [
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
]

comparison['Month'] = pd.Categorical(
    comparison['Month'],
    categories=month_order,
    ordered=True
)

comparison = comparison.sort_values('Month')

# ===============================
# 6️⃣ Save final CSV
# ===============================
comparison.to_csv(
    r'C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\model\monthly_profit_revenue_comparison_2025.csv',
    index=False
)

print("2025 Actual vs AI Revenue & Profit comparison saved!")
print(comparison)



✅ 2025 Actual vs AI Revenue & Profit comparison saved!
        Month  Revenue_Actual  Profit_Actual  Revenue_Predicted  \
4     January          4389.0         1008.0        5018.476244   
3    February          3524.5          788.5        3802.877554   
7       March          5580.0         1308.0        6241.322660   
0       April          4752.0         1056.0        5360.779200   
8         May          6227.0         1417.0        7143.552305   
6        June          6972.0         1596.0        7900.687645   
5        July          7936.0         1767.0        9068.907659   
1      August          7148.5         1580.5        7781.946989   
11  September          5129.0         1104.0        5642.757619   
10    October          4752.0         1056.0        5300.450437   
9    November          3850.0          830.0        4345.573775   
2    December          6277.5         1363.5        7044.035348   

    Profit_Predicted  Profit_Difference  Revenue_Difference  
4        15